In [ ]:
import os
import json
import re
import unicodedata
import numpy as np
from difflib import SequenceMatcher
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from peft import PeftConfig, PeftModel
import warnings
warnings.filterwarnings('ignore')
os.environ["HF_HUB_OFFLINE"] = "1"

# ==========================================
# 1. CONFIGURATION

BASE_DIR = r"D:\student1402\negar\final_research"
RE_DIR   = os.path.join(BASE_DIR, "phase2_model_pipeline_aware_abstrcat_clean")
NER_DIR = os.path.join(BASE_DIR, "models", "ner_abstract_optimized_clean") 
BASE_MODEL_DIR = r"C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d"
TEST_PATH        = os.path.join(BASE_DIR, "data", "curated_test_set.json")
OUTPUT_JSON_PATH = os.path.join(BASE_DIR, "e2e_final_predictions_pipeline_aware_clean.json")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


c:\Users\UMZ\anaconda3\envs\pubmedbert\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:

# ==========================================
# 2. LOAD MODELS
# ==========================================
print("\nLoading NER Model...")
ner_tokenizer = AutoTokenizer.from_pretrained(NER_DIR, use_fast=True, local_files_only=True)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_DIR, local_files_only=True).to(DEVICE)
ner_model.eval()

print("Loading RE Model (LoRA)...")
peft_config = PeftConfig.from_pretrained(RE_DIR)

# FIX: Try loading tokenizer from RE_DIR first to catch special tokens
try:
    re_tokenizer = AutoTokenizer.from_pretrained(RE_DIR, local_files_only=True)
except:
    re_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)

base_re_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_DIR, num_labels=2, local_files_only=True)
re_model = PeftModel.from_pretrained(base_re_model, RE_DIR).to(DEVICE)
re_model.eval()

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_raw = json.load(f)

# ==========================================
# 3. GENERIC FILTER & RANGE EXPANSION
# ==========================================
GENERIC_STOPWORDS = {
    "compound", "compounds", "derivative", "derivatives", "analogue", "analogues", 
    "extract", "extracts", "fraction", "fractions", "mixture", "mixtures", 
    "synthetic", "racemate", "racemates", "acarbose", "lipopolysaccharide", "lps",
    "prostaglandin", "pge2", "pge(2)", "acid", "acids", "oil", "oils", "solvent",
    "methanol", "ethanol", "ethyl acetate", "chloroform", "water", "buffer"
}

def is_valid_chemical(chem_str):
    c_lower = chem_str.lower().strip()
    if c_lower in GENERIC_STOPWORDS: return False
    # Catch "compound 6", "racemate 4", "12A analogue"
    if re.match(r'^(compound|racemate|analogue|derivative)s?\s+[a-z0-9\-]+$', c_lower): return False
    if "analogue" in c_lower or "derivative" in c_lower or "synthetic" in c_lower: return False
    return True

def expand_chemical_range(chem_str):
    clean_str = re.sub(r'\s*\([\d\-\s,a-zA-Z]+$', '', chem_str).strip()
    expanded_list = []
    
    match_alpha = re.match(r'^([a-zA-Z\-_]+?)(?:s)?\s+([A-Z])-([A-Z])$', clean_str, re.IGNORECASE)
    if match_alpha:
        base_name = match_alpha.group(1).capitalize()
        for c in range(ord(match_alpha.group(2).upper()), ord(match_alpha.group(3).upper()) + 1):
            expanded_list.append(f"{base_name} {chr(c)}")
        return expanded_list

    match_num = re.match(r'^([a-zA-Z\-_]+?)(?:s)?\s+(\d+)-(\d+)$', clean_str, re.IGNORECASE)
    if match_num:
        base_name = match_num.group(1).capitalize()
        s_num, e_num = int(match_num.group(2)), int(match_num.group(3))
        if s_num < e_num and (e_num - s_num) < 30:
            for i in range(s_num, e_num + 1):
                expanded_list.append(f"{base_name} {i}")
            return expanded_list
    return [clean_str]

def inject_markers(abstract, org_str, chem_str):
    i_org = abstract.find(org_str)
    i_chem = abstract.find(chem_str)
    if i_org == -1: i_org = abstract.lower().find(org_str.lower())
    if i_chem == -1: i_chem = abstract.lower().find(chem_str.lower())
    if i_org == -1 or i_chem == -1: return None
    if max(i_org, i_chem) < min(i_org + len(org_str), i_chem + len(chem_str)): return None 
    
    s = abstract
    if i_org > i_chem:
        s = s[:i_org] + "[O]" + abstract[i_org:i_org+len(org_str)] + "[/O]" + s[i_org+len(org_str):]
        s = s[:i_chem] + "[C]" + abstract[i_chem:i_chem+len(chem_str)] + "[/C]" + s[i_chem+len(chem_str):]
    else:
        s = s[:i_chem] + "[C]" + abstract[i_chem:i_chem+len(chem_str)] + "[/C]" + s[i_chem+len(chem_str):]
        s = s[:i_org] + "[O]" + abstract[i_org:i_org+len(org_str)] + "[/O]" + s[i_org+len(org_str):]
    return s

# ==========================================
# 4. NER PIPELINE 
# ==========================================
def run_ner(abstract):
    if not abstract: return set(), set()
    sentences = re.split(r'(?<=\.) ', abstract)
    chunks, current = [], ""
    for sent in sentences:
        test_str = (current + " " + sent).strip()
        if len(ner_tokenizer.encode(test_str)) <= 400: current = test_str
        else:
            if current: chunks.append(current)
            current = sent
    if current: chunks.append(current)
        
    pred_orgs, pred_chems = set(), set()
    
    for chunk in chunks:
        inputs = ner_tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512, return_offsets_mapping=True)
        with torch.no_grad():
            outputs = ner_model(inputs["input_ids"].to(DEVICE), attention_mask=inputs["attention_mask"].to(DEVICE))
            
        offsets = inputs["offset_mapping"].squeeze(0).cpu().tolist()
        word_ids = inputs.word_ids(batch_index=0)
        probs = torch.softmax(outputs.logits, dim=-1).squeeze(0).cpu().numpy()
        
        word_max_probs, word_bounds = {}, {}
        for idx, w_id in enumerate(word_ids):
            if w_id is None: continue
            start_char, end_char = offsets[idx]
            if start_char == end_char: continue
            if w_id not in word_max_probs:
                word_max_probs[w_id] = {'org': 0.0, 'chem': 0.0}
                word_bounds[w_id] = [start_char, end_char]
            word_max_probs[w_id]['org'] = max(word_max_probs[w_id]['org'], probs[idx, 1] + probs[idx, 2])
            word_max_probs[w_id]['chem'] = max(word_max_probs[w_id]['chem'], probs[idx, 3] + probs[idx, 4])
            word_bounds[w_id][1] = end_char 
            
        raw_spans = []
        for w_id, p in word_max_probs.items():
            if p['org'] >= NER_ORG_THRESH and p['org'] > p['chem']: raw_spans.append(('ORG', word_bounds[w_id][0], word_bounds[w_id][1]))
            elif p['chem'] >= NER_CHEM_THRESH and p['chem'] > p['org']: raw_spans.append(('CHEM', word_bounds[w_id][0], word_bounds[w_id][1]))
            
        if not raw_spans: continue
        
        merged_spans = [raw_spans[0]]
        for i in range(1, len(raw_spans)):
            prev_label, prev_start, prev_end = merged_spans[-1]
            curr_label, curr_start, curr_end = raw_spans[i]
            
            gap = chunk[prev_end:curr_start]
            # IUPAC No-Space rule:
            if prev_label == 'CHEM' and curr_label == 'CHEM' and " " not in gap:
                merged_spans[-1] = (prev_label, prev_start, curr_end)
            elif prev_label == curr_label:
                if len(gap.strip()) == 0 or re.match(r'^[\-\+\[\]\(\)]+$', gap.strip()):
                    merged_spans[-1] = (prev_label, prev_start, curr_end)
                else:
                    merged_spans.append((curr_label, curr_start, curr_end))
            else:
                merged_spans.append((curr_label, curr_start, curr_end))
                
        for label, start, end in merged_spans:
            ent_str = chunk[start:end].strip('.,;:)(" \'-')
            ent_str = re.sub(r'\s*\([\d\-\s,a-zA-Z]+$', '', ent_str).strip()
            
            # Split internal commas (e.g. "iso-T-2, T-2 triol" -> two entities)
            sub_entities = re.split(r',\s+|\s+and\s+', ent_str)
            for sub_ent in sub_entities:
                sub_ent = sub_ent.strip('.,;:)(" \'-')
                # Strip dangling conjunctions like "and E. sanguinea"
                sub_ent = re.sub(r'^(and|or|the|a|an)\s+', '', sub_ent, flags=re.IGNORECASE)
                
                if len(sub_ent) >= 3:
                    if label == 'ORG': pred_orgs.add(sub_ent)
                    elif label == 'CHEM' and is_valid_chemical(sub_ent): pred_chems.add(sub_ent)
                
    return pred_orgs, pred_chems

# ==========================================
# 5. EVALUATION MATCHING LOGIC
# ==========================================
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).lower().strip()
    return re.sub(r'\s+', ' ', text)

def match_organisms(pred_norm, gold_norm):
    if pred_norm == gold_norm: return True
    if len(pred_norm.split()) == 2 and len(gold_norm.split()) >= 2:
        p_parts, g_parts = pred_norm.replace('.', '').split(), gold_norm.split()
        if p_parts[0][0] == g_parts[0][0] and p_parts[1] == g_parts[1]: return True
    if pred_norm in gold_norm or gold_norm in pred_norm:
        if len(pred_norm) / len(gold_norm) >= 0.50: return True
    if SequenceMatcher(None, pred_norm, gold_norm).ratio() >= 0.85: return True
    return False

def match_chemicals(pred_norm, gold_norm):
    if pred_norm == gold_norm: return True
    if pred_norm in gold_norm or gold_norm in pred_norm:
        if min(len(pred_norm), len(gold_norm)) / max(len(pred_norm), len(gold_norm)) >= 0.50: return True
    if SequenceMatcher(None, pred_norm, gold_norm).ratio() >= 0.85: return True
    return False

def is_pair_match(p_org, p_chem, available_gold_pairs):
    p_org_n = normalize(p_org)
    p_chem_n = normalize(p_chem)
    
    for g_org, g_chem in available_gold_pairs:
        if p_org_n == normalize(g_org) and p_chem_n == normalize(g_chem): return (g_org, g_chem)
    for g_org, g_chem in available_gold_pairs:
        if p_org_n == normalize(g_org) and match_chemicals(p_chem_n, normalize(g_chem)): return (g_org, g_chem)
    for g_org, g_chem in available_gold_pairs:
        if match_organisms(p_org_n, normalize(g_org)) and match_chemicals(p_chem_n, normalize(g_chem)): return (g_org, g_chem)
            
    return None

# ==========================================
# 6. RUN THE E2E SYSTEM
# ==========================================
print("\nStarting End-to-End Pipeline...")
final_output = {}
tp, fp, fn = 0, 0, 0

for pmid, doc in tqdm(test_raw.items(), desc="Processing Abstracts"):
    abstract = doc.get("AbstractText", "")
    if not abstract: continue
        
    org_dict = {o["id"]: o["label"] for o in doc.get("organisms", [])}
    chem_dict = {c["id"]: c["label"] for c in doc.get("chemicals", [])}
    gold_pairs = [(org_dict.get(r[0]), chem_dict.get(r[1])) for r in doc.get("relations", []) if r[0] in org_dict and r[1] in chem_dict]
    golds_available = list(gold_pairs)
    
    predicted_pairs_expanded = []
    raw_re_extractions = []
    
    pred_orgs, pred_chems = run_ner(abstract)
    
    for org in pred_orgs:
        for chem in pred_chems:
            marked_text = inject_markers(abstract, org, chem)
            if not marked_text: continue
            
            # Using max_length=512 to prevent truncation blindness safely
            inputs = re_tokenizer([marked_text], padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
            
            with torch.no_grad():
                logits = re_model(**inputs).logits
                prob = torch.softmax(logits, dim=1)[:, 1].item()
                
            if prob >= RE_THRESH:
                raw_re_extractions.append({"org": org, "chem_raw": chem, "prob": round(prob, 4)})
                expanded_chems = expand_chemical_range(chem)
                for exp_chem in expanded_chems:
                    predicted_pairs_expanded.append({"org": org, "chem": exp_chem})
                    
    # Sort predictions so exact/longer texts are processed first
    predicted_pairs_expanded.sort(key=lambda x: len(x["chem"]), reverse=True)
                    
    tp_list, fp_list = [], []
    for pair in predicted_pairs_expanded:
        p_org, p_chem = pair["org"], pair["chem"]
        matched_gold_pair = is_pair_match(p_org, p_chem, golds_available)
        
        if matched_gold_pair:
            tp += 1
            tp_list.append({"predicted": pair, "matched_gold": matched_gold_pair})
            golds_available.remove(matched_gold_pair)
        else:
            fp += 1
            fp_list.append(pair)
            
    fn += len(golds_available)
    final_output[pmid] = {
        "RAW_RE_EXTRACTIONS": raw_re_extractions,
        "TRUE_POSITIVES_PAIRS": tp_list,
        "FALSE_POSITIVES_PAIRS": fp_list,
        "FALSE_NEGATIVES_PAIRS": [{"org": g[0], "chem": g[1]} for g in golds_available]
    }

# ==========================================
# 7. PRINT AND SAVE RESULTS
# ==========================================
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score  = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "="*50)
print("🏆 FINAL END-TO-END PIPELINE PERFORMANCE 🏆")
print("="*50)
print(f"Total True Positives (Pairs) : {tp}")
print(f"Total False Positives (Pairs): {fp}")
print(f"Total False Negatives (Pairs): {fn}")
print("-" * 50)
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1_score:.4f}")
print("="*50)

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4, ensure_ascii=False)


Loading NER Model...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading RE Model (LoRA)...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at C:\Users\UMZ\.cache\huggingface\hub\models--microsoft--BiomedNLP-PubMedBERT-base-uncased-abstract\snapshots\d673b8835373c6fa116d6d8006b33d48734e305d and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting End-to-End Pipeline...


Processing Abstracts: 100%|██████████| 200/200 [00:41<00:00,  4.87it/s]


🏆 FINAL END-TO-END PIPELINE PERFORMANCE 🏆
Total True Positives (Pairs) : 816
Total False Positives (Pairs): 198
Total False Negatives (Pairs): 677
--------------------------------------------------
Precision : 0.8047
Recall    : 0.5466
F1-Score  : 0.6510
